In [1]:
import numpy as np
import cvxpy as cp
import mosek
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [226]:
def makeset (A, B):    # we assume A is non-empty
    N = len(A)
    added = []
    for i in range(N):
        new = B[0:i+1]
        N_sets = len(A[i])
        for k in range(N_sets-1):
            if len(np.intersect1d(A[i][k],new))==len(new):
                break
            if k == N_sets-2:
                A[i].append(new)
                added.append(new)
    return(A,added)

def countsets(sets):
    m = len(sets)
    count = 0
    for k in range(m):
        count = count + len(sets[k])
    return(count)

def convertlist(sets):
    Output = []
    for temp in sets:
        for elem in temp:
            Output.append(elem)
    return(Output)

def solvenominal (sets,p,R,r,m,r_f,c):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable(M,N)
    lbda = cp.Variable(M, nonneg = True)
    a = cp.Variable(I)
    alpha = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N, nonneg = True)
    z2 = 0
    z4 = 0
    constraints = []
    for i in range(N):
        lbdasum = 0
        for j in range(M):
            z1 = -cp.min(v[j][sets[j]])*(1-m)+lbda[j]
            z2 = z2 + cp.pos(z1)
            if i in sets[j]:
                cosntraints.append(v[j][i] >= 0)
                lbdasum = lbdasum + lbda[j]
            else:
                constraints.append(v[j][i] <= 0)
        constraints.append(-a.T @ R - lbdasum <= 0)
        z4 = z4 + p[i]*t[i]
        constraints.append(-alpha + cp.sum(v[0:M:1,i]) + cp.kl_div(gamma, t[i]) + gamma - t[i] <= 0)
    
    constraints.append(alpha + gamma * r - (1-cp.sum(a))*r_f + z4 -1 + z2 <= c)
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve()
    return(prob.value,v.value,a.value)
    
    
def robustcheck(a,R,r,c,p,m):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    sets = []
    for i in range(N):
        sets.append()
    q_b = cp.Variable(N)
    q = cp.Variable(N)
    constraints = [q >= 0, q_b >= 0, cp.sum(q) == 1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(z1-v <= 0)
        phi_cons = phi_cons -(cp.entr(q[i]) + q[i]*np.log(p[i]))
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve()
    return(prob.value <= c)
    
                
    

In [155]:
def Robust_portfolio_h3_pos (x, p, r, r_f, m, c):
    N = len(x)
    lbda = cp.Variable(N)
    v = cp.Variable((N,N))
    t = cp.Variable(N, nonneg = True)
    alpha = cp.Variable(1)
    gamma = cp.Variable(1, nonneg = True)
    a = cp.Variable(1, nonneg = True)
    z2 = 0
    z4 = 0
    constraints = []
    for i in range(N-1):
        constraints.append(v[i][0:(N-i-1)] <= 0)
        constraints.append(v[i][(N-i-1):N] >= 0)
        constraints.append(-a*x[i]-cp.sum(lbda[(N-i-1):N]) <= 0)
    constraints.append(-a*x[N-1]-cp.sum(lbda) <= 0)
    constraints.append(v[N-1] >= 0)
    for j in range(N):
        z1 = -cp.min(v[j,(N-1-j):N])*(1-m)+lbda[j]
        z2 = z2 + cp.pos(z1)
    for i in range(N):
        z4 = z4 + p[i]*t[i]
        constraints.append(-alpha + cp.sum(v[0:N:1,i]) + cp.kl_div(gamma, t[i]) + gamma - t[i] <= 0) 
    constraints.append(alpha + gamma * r - (1-a)*r_f + z4 -1 + z2 <= c)
    #constraints.append(a <= 1)
    obj = cp.Maximize(a*x.T @ p + (1-a)*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve()
    return(prob.value,v.value,a.value) 

([[[1], [2]], [[1, 2], [2, 3]], [[1, 2, 3], [2, 3, 4]]], [])
[[[1], [2]], [[1, 2], [2, 3]], [[1, 2, 3], [2, 3, 4]]]


In [210]:
A=[[[1],[2],[3]],[[1,2],[3,4]],[[1,2,3]],[[1,2,3,4]]]
B = [2,1,3,4]
makeset(A,B)

([[[1], [2], [3]], [[1, 2], [3, 4]], [[1, 2, 3]], [[1, 2, 3, 4]]], [])

In [225]:
x= np.array([[5,8,1],[1,0,2],[2,1,3]])
y = np.array([3,1,2])
print(x.dot(y))
print(np.argsort(x.dot(y)))

[25  7 13]
[1 2 0]
